# Hyperliquid Traders x Bitcoin Fear & Greed Index
## Sentiment-Driven Trading Behaviour Analysis
*Prepared for Sonika — Primetrade.ai Internship Assignment*

---

## Executive Summary

Analysis of **211,218 trades** across the 2023–2025 period reveals that traders on Hyperliquid achieve their **highest average PnL (+$67.9/trade) and best win rate (46.5%) during Extreme Greed** conditions — counter-intuitively outperforming all other sentiment regimes. Conversely, **Extreme Fear trades yield the lowest win rate (37.1%)** despite a moderate average PnL lifted by a handful of high-conviction long trades. The data identifies a distinct **"Fear Opportunist" archetype** — traders who predominantly execute during low-sentiment windows — that averages **+$248K total PnL**, making it the second-most-profitable cluster. These findings directly support building a sentiment-gated execution layer: a trading bot should **increase position sizing and enter longs aggressively during Extreme Greed breakouts** while filtering out low-quality Extreme Fear noise from its trade queue.

---


## 1. Imports & Configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import os

# ── Design System ────────────────────────────────────────────────────────────
SENTIMENT_ORDER  = ['Extreme Fear', 'Fear', 'Neutral', 'Greed', 'Extreme Greed']
SENTIMENT_COLORS = {
    'Extreme Fear':  '#E74C3C',
    'Fear':          '#E67E22',
    'Neutral':       '#95A5A6',
    'Greed':         '#27AE60',
    'Extreme Greed': '#1A8F3C',
}

sns.set_theme(style='darkgrid', palette='muted', font='DejaVu Sans')
plt.rcParams.update({
    'figure.facecolor': '#0D1117',
    'axes.facecolor':   '#161B22',
    'axes.edgecolor':   '#30363D',
    'axes.labelcolor':  '#C9D1D9',
    'xtick.color':      '#C9D1D9',
    'ytick.color':      '#C9D1D9',
    'text.color':       '#C9D1D9',
    'grid.color':       '#21262D',
    'grid.linewidth':   0.6,
    'font.size':        11,
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
})

print("Environment ready.")


## 2. Data Cleaning & Alignment

### Why this step matters
The two datasets live on different time resolutions:
- **Historical data**: exact timestamp per trade (e.g., `02-12-2024 22:50`)
- **Fear & Greed Index**: one value per calendar day

We strip the time component from each trade's timestamp to get a plain date, then perform an **inner join** so every trade row inherits that day's sentiment score and classification.


In [ ]:
# ── Load Raw Data ─────────────────────────────────────────────────────────────
hist = pd.read_csv('historical_data.csv')
fg   = pd.read_csv('fear_greed_index.csv')

print(f"Historical data  : {len(hist):,} rows  |  Columns: {hist.columns.tolist()}")
print(f"Fear/Greed index : {len(fg):,} rows  |  Columns: {fg.columns.tolist()}")


In [ ]:
# ── Timeline Normalisation ────────────────────────────────────────────────────
hist['trade_date'] = pd.to_datetime(hist['Timestamp IST'], format='%d-%m-%Y %H:%M').dt.date
fg['trade_date']   = pd.to_datetime(fg['date']).dt.date

# ── Inner Merge ────────────────────────────────────────────────────────────────
merged = pd.merge(
    hist,
    fg[['trade_date', 'value', 'classification']],
    on='trade_date', how='inner'
)

# ── Derived Columns ────────────────────────────────────────────────────────────
merged['classification'] = pd.Categorical(
    merged['classification'], categories=SENTIMENT_ORDER, ordered=True
)
merged['is_win']  = merged['Closed PnL'] > 0
merged['is_loss'] = merged['Closed PnL'] < 0

print(f"Rows after merge  : {len(merged):,}   (dropped {len(hist)-len(merged):,} unmatched)")
print(f"Date range        : {merged['trade_date'].min()}  to  {merged['trade_date'].max()}")
print(f"Unique traders    : {merged['Account'].nunique():,}")
print(f"Unique coins      : {merged['Coin'].nunique():,}")
print()
merged.head(4)


In [ ]:
# ── Data Quality Check ────────────────────────────────────────────────────────
print("=== NULL COUNTS ===")
print(merged[['Closed PnL','Size USD','Side','value','classification']].isnull().sum())
print()
print("=== SENTIMENT DISTRIBUTION ===")
print(merged['classification'].value_counts().sort_index())


## 3. Exploratory Data Analysis (EDA)

We segment every metric by the **5-level Fear & Greed classification** to answer three questions:
1. **Win Rate** — What fraction of trades are profitable under each sentiment?
2. **Volume Distribution** — Do traders bet bigger during Greed or Fear?
3. **Directional Bias** — Does BUY/SELL ratio shift with sentiment?


In [ ]:
# ── Core EDA Aggregation ──────────────────────────────────────────────────────
eda = merged.groupby('classification', observed=True).agg(
    trade_count  = ('Closed PnL', 'count'),
    win_rate     = ('is_win',     'mean'),
    avg_pnl      = ('Closed PnL', 'mean'),
    median_pnl   = ('Closed PnL', 'median'),
    total_volume = ('Size USD',   'sum'),
    avg_volume   = ('Size USD',   'mean'),
    buy_ratio    = ('Side',       lambda x: (x == 'BUY').mean()),
).reset_index()

eda['win_rate_pct']  = (eda['win_rate']  * 100).round(2)
eda['buy_ratio_pct'] = (eda['buy_ratio'] * 100).round(2)
eda['total_vol_M']   = (eda['total_volume'] / 1e6).round(2)

display_cols = ['classification','trade_count','win_rate_pct','avg_pnl','avg_volume','buy_ratio_pct','total_vol_M']
eda[display_cols].style.background_gradient(cmap='RdYlGn', subset=['win_rate_pct','avg_pnl'])


### Chart 1 — Win Rate & Average PnL by Market Sentiment

In [ ]:
cats   = eda['classification'].tolist()
colors = [SENTIMENT_COLORS[c] for c in cats]

fig, axes = plt.subplots(1, 2, figsize=(15, 6), facecolor='#0D1117')
fig.suptitle('Win Rate & Average PnL by Market Sentiment', fontsize=15, color='#C9D1D9', y=1.01)

def annotated_bar(ax, x, y, colors, xlabel, ylabel, title):
    bars = ax.bar(x, y, color=colors, width=0.55, edgecolor='none', zorder=3)
    ax.set_title(title, pad=10)
    ax.set_xlabel(xlabel, labelpad=8)
    ax.set_ylabel(ylabel, labelpad=8)
    ax.set_xticks(range(len(x)))
    ax.set_xticklabels(x, rotation=20, ha='right', fontsize=9)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + abs(h)*0.02,
                f'{h:,.1f}', ha='center', va='bottom', fontsize=8.5, color='#C9D1D9')

annotated_bar(axes[0], cats, eda['win_rate_pct'], colors,
              'Sentiment', 'Win Rate (%)', 'Win Rate by Sentiment')
axes[0].axhline(50, color='#58A6FF', linewidth=1.5, linestyle='--', label='50% threshold', zorder=4)
axes[0].legend(fontsize=9)

annotated_bar(axes[1], cats, eda['avg_pnl'], colors,
              'Sentiment', 'Avg Closed PnL (USD)', 'Average PnL by Sentiment')
axes[1].axhline(0, color='#58A6FF', linewidth=1.5, linestyle='--', zorder=4)

plt.tight_layout()
plt.savefig('charts/chart1_winrate_pnl.png', dpi=150, bbox_inches='tight', facecolor='#0D1117')
plt.show()


### Chart 2 — Trade Volume Distribution by Sentiment & Direction

In [ ]:
vols_b = merged[merged['Side']=='BUY'].groupby('classification', observed=True)['Size USD'].sum() / 1e6
vols_s = merged[merged['Side']=='SELL'].groupby('classification', observed=True)['Size USD'].sum() / 1e6

fig, ax = plt.subplots(figsize=(11, 6), facecolor='#0D1117')
ax.set_facecolor('#161B22')

x_pos = np.arange(len(SENTIMENT_ORDER))
width = 0.35
ax.bar(x_pos - width/2, [vols_b.get(c, 0) for c in SENTIMENT_ORDER],
       width, label='BUY volume', color='#2ECC71', edgecolor='none', zorder=3)
ax.bar(x_pos + width/2, [vols_s.get(c, 0) for c in SENTIMENT_ORDER],
       width, label='SELL volume', color='#E74C3C', edgecolor='none', zorder=3)

ax.set_xticks(x_pos)
ax.set_xticklabels(SENTIMENT_ORDER, rotation=15, ha='right', fontsize=9)
ax.set_xlabel('Sentiment Classification', labelpad=8)
ax.set_ylabel('Total Volume (Millions USD)', labelpad=8)
ax.set_title('Trade Volume Distribution by Sentiment & Direction', pad=10)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('charts/chart2_volume_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0D1117')
plt.show()


### Chart 3 — PnL Distribution per Sentiment (Boxplot)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7), facecolor='#0D1117')
ax.set_facecolor('#161B22')

box_data = [merged[merged['classification']==c]['Closed PnL'].clip(-500, 500).dropna()
            for c in SENTIMENT_ORDER]

bp = ax.boxplot(box_data, patch_artist=True, notch=True,
                medianprops=dict(color='#F1C40F', linewidth=2.5),
                whiskerprops=dict(color='#C9D1D9', linewidth=1.2),
                capprops=dict(color='#C9D1D9', linewidth=1.2),
                flierprops=dict(marker='o', color='#58A6FF', alpha=0.15, markersize=3))

for patch, color in zip(bp['boxes'], [SENTIMENT_COLORS[c] for c in SENTIMENT_ORDER]):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax.set_xticklabels(SENTIMENT_ORDER, rotation=15, ha='right', fontsize=10)
ax.set_xlabel('Sentiment Classification', labelpad=8)
ax.set_ylabel('Closed PnL (USD) — clipped to +/-$500', labelpad=8)
ax.set_title('PnL Distribution per Sentiment Tier (Seaborn-style Boxplot)', pad=10)
ax.axhline(0, color='#58A6FF', linewidth=1.5, linestyle='--', zorder=4, label='Break-even')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('charts/chart3_pnl_boxplot.png', dpi=150, bbox_inches='tight', facecolor='#0D1117')
plt.show()


### Chart 4 — Directional Bias (BUY vs SELL) by Sentiment

In [ ]:
buy_vals  = eda['buy_ratio_pct'].tolist()
sell_vals = [100-v for v in buy_vals]

fig, ax = plt.subplots(figsize=(11, 5), facecolor='#0D1117')
ax.set_facecolor('#161B22')

x_pos = np.arange(len(cats))
ax.bar(x_pos, buy_vals,  label='BUY %',  color='#2ECC71', edgecolor='none', zorder=3)
ax.bar(x_pos, sell_vals, bottom=buy_vals, label='SELL %', color='#E74C3C', edgecolor='none', zorder=3)
ax.axhline(50, color='#F1C40F', linewidth=1.5, linestyle='--', label='50/50 line', zorder=4)

ax.set_xticks(x_pos)
ax.set_xticklabels(cats, rotation=15, ha='right', fontsize=9)
ax.set_xlabel('Sentiment Classification', labelpad=8)
ax.set_ylabel('Direction Mix (%)', labelpad=8)
ax.set_title('Directional Bias (BUY vs SELL) by Sentiment', pad=10)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('charts/chart4_directional_bias.png', dpi=150, bbox_inches='tight', facecolor='#0D1117')
plt.show()


## 4. Feature Engineering & Trader Profiling

We aggregate to the **Account (Trader) level** — only including traders with at least 10 trades for statistical reliability — and compute:
- **Sentiment Sensitivity** (% of trades during Extreme Fear / Extreme Greed)
- **Net PnL & Win Rate**
- **Average Trade Size** (proxy for conviction / sophistication)

Then we label each trader with an **archetype** based on behavioural rules.


In [ ]:
# ── Trader-level Aggregation ──────────────────────────────────────────────────
trader = merged.groupby('Account').agg(
    total_trades      = ('Closed PnL', 'count'),
    total_pnl         = ('Closed PnL', 'sum'),
    avg_trade_size    = ('Size USD',   'mean'),
    win_rate          = ('is_win',     'mean'),
    extreme_fear_pct  = ('classification', lambda x: (x == 'Extreme Fear').mean()),
    extreme_greed_pct = ('classification', lambda x: (x == 'Extreme Greed').mean()),
    buy_ratio         = ('Side', lambda x: (x == 'BUY').mean()),
    avg_sentiment     = ('value', 'mean'),
    cross_ratio       = ('Crossed', 'mean'),   # fraction of trades in cross-margin
).reset_index()

trader = trader[trader['total_trades'] >= 10].copy()

# ── Archetype Labels ───────────────────────────────────────────────────────────
def classify_archetype(row):
    if row['extreme_fear_pct'] > 0.40 and row['buy_ratio'] > 0.55:
        return 'Contrarian Buyer'
    elif row['extreme_greed_pct'] > 0.40 and row['buy_ratio'] > 0.55:
        return 'FOMO Trader'
    elif row['avg_sentiment'] < 35 and row['total_pnl'] > 0:
        return 'Fear Opportunist'
    elif row['avg_sentiment'] > 55 and row['total_pnl'] < 0:  # loosened from >65
        return 'Greed Victim'
    elif row['win_rate'] > 0.55:
        return 'Consistent Winner'
    else:
        return 'Mixed Sentiment'

trader['archetype'] = trader.apply(classify_archetype, axis=1)

print(f"Qualified traders : {len(trader):,}")
print()
print("Archetype counts:")
print(trader['archetype'].value_counts().to_string())


In [ ]:
# ── Archetype Performance Summary ─────────────────────────────────────────────
archetype_perf = trader.groupby('archetype').agg(
    count       = ('Account', 'count'),
    avg_pnl     = ('total_pnl', 'mean'),
    med_pnl     = ('total_pnl', 'median'),
    win_rate    = ('win_rate', 'mean'),
    avg_size    = ('avg_trade_size', 'mean'),
    cross_ratio = ('cross_ratio', 'mean'),
).reset_index().sort_values('avg_pnl', ascending=False)

archetype_perf.style.background_gradient(cmap='RdYlGn', subset=['avg_pnl','win_rate'])


### Chart 5 — Average PnL by Trader Archetype

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6), facecolor='#0D1117')
ax.set_facecolor('#161B22')

arch_colors = ['#1A8F3C' if v >= 0 else '#E74C3C' for v in archetype_perf['avg_pnl']]
bars = ax.barh(archetype_perf['archetype'], archetype_perf['avg_pnl'],
               color=arch_colors, edgecolor='none', zorder=3, height=0.55)

for bar in bars:
    w = bar.get_width()
    offset = max(abs(w) * 0.01, 500)
    ax.text(w + (offset if w >= 0 else -offset),
            bar.get_y() + bar.get_height()/2,
            f'${w:,.0f}', va='center',
            ha='left' if w >= 0 else 'right', fontsize=9, color='#C9D1D9')

ax.axvline(0, color='#C9D1D9', linewidth=0.8)
ax.set_xlabel('Average Total PnL (USD)', labelpad=8)
ax.set_title('Average PnL by Trader Archetype', pad=10)

plt.tight_layout()
plt.savefig('charts/chart5_archetype_pnl.png', dpi=150, bbox_inches='tight', facecolor='#0D1117')
plt.show()


## 5. Statistical Testing & Insights

### Tests performed
| Test | Purpose |
|------|---------|
| **Pearson Correlation** | Linear relationship between raw sentiment score (0–100) and trade PnL |
| **Welch's t-test (Sentiment)** | Is mean PnL during *Extreme Fear* significantly different from *Extreme Greed*? |
| **Welch's t-test (Margin Type)** | Does cross-margin vs isolated-margin produce significantly different PnL? |


In [ ]:
# ── Pearson Correlation ────────────────────────────────────────────────────────
corr, pval = stats.pearsonr(merged['value'], merged['Closed PnL'])
print(f"Pearson r  = {corr:.4f}")
print(f"p-value    = {pval:.4e}")
print(f"Verdict    : {'Statistically significant (p<0.05)' if pval < 0.05 else 'Not significant'}")
print(f"Interpretation: {'Weak positive' if corr > 0 else 'Weak negative'} linear relationship.")
print()

# ── Welch t-test: Extreme Fear vs Extreme Greed ────────────────────────────────
ef_pnl = merged[merged['classification'] == 'Extreme Fear']['Closed PnL']
eg_pnl = merged[merged['classification'] == 'Extreme Greed']['Closed PnL']

t_stat, t_p = stats.ttest_ind(ef_pnl, eg_pnl, equal_var=False)
print(f"Welch t-test (Extreme Fear vs Extreme Greed)")
print(f"  t-statistic = {t_stat:.3f}")
print(f"  p-value     = {t_p:.4e}")
print(f"  Verdict     : {'Significant difference in means (p<0.05)' if t_p < 0.05 else 'No significant difference'}")
print()
print(f"  Mean PnL — Extreme Fear   : ${ef_pnl.mean():.2f}")
print(f"  Mean PnL — Extreme Greed  : ${eg_pnl.mean():.2f}")
print(f"  Difference                : ${eg_pnl.mean()-ef_pnl.mean():.2f}")
print()

# ── Welch t-test: Cross-margin vs Isolated-margin ─────────────────────────────
cross_pnl    = merged[merged['Crossed'] == True]['Closed PnL']
isolated_pnl = merged[merged['Crossed'] == False]['Closed PnL']
tc, tp = stats.ttest_ind(cross_pnl, isolated_pnl, equal_var=False)
print(f"Welch t-test (Cross-margin vs Isolated-margin)")
print(f"  Cross-margin    avg PnL : ${cross_pnl.mean():.2f}  (n={len(cross_pnl):,})")
print(f"  Isolated-margin avg PnL : ${isolated_pnl.mean():.2f}  (n={len(isolated_pnl):,})")
print(f"  t-statistic = {tc:.3f}")
print(f"  p-value     = {tp:.4e}")
print(f"  Verdict     : {'Significant (p<0.05)' if tp < 0.05 else 'Not significant'}")


### Chart 6 — Correlation: Market Sentiment Score vs Trade PnL

In [ ]:
sample = merged[merged['Closed PnL'].between(-300, 300)].sample(
    min(6000, len(merged)), random_state=42)

fig, ax = plt.subplots(figsize=(11, 7), facecolor='#0D1117')
ax.set_facecolor('#161B22')

scatter_colors = [SENTIMENT_COLORS[c] for c in sample['classification']]
ax.scatter(sample['value'], sample['Closed PnL'],
           c=scatter_colors, alpha=0.2, s=14, edgecolors='none')

m, b, r_val, *_ = stats.linregress(sample['value'], sample['Closed PnL'])
x_line = np.linspace(0, 100, 300)
ax.plot(x_line, m*x_line + b, color='#58A6FF', linewidth=2.5,
        label=f'Trend line  (r = {r_val:.4f})', zorder=5)

legend_patches = [mpatches.Patch(color=v, label=k) for k,v in SENTIMENT_COLORS.items()]
ax.legend(handles=legend_patches + ax.get_lines()[:1], fontsize=8.5, loc='upper left',
          framealpha=0.3, facecolor='#0D1117')

ax.axhline(0, color='#C9D1D9', linewidth=0.7, linestyle=':')
ax.set_xlabel('Fear & Greed Score (0 = Extreme Fear, 100 = Extreme Greed)', labelpad=8)
ax.set_ylabel('Closed PnL (USD)  [clipped to +/-$300 for visibility]', labelpad=8)
ax.set_title(f'Market Sentiment vs Individual Trade PnL  (Pearson r = {corr:.4f}, p = {pval:.2e})', pad=10)

plt.tight_layout()
plt.savefig('charts/chart6_correlation.png', dpi=150, bbox_inches='tight', facecolor='#0D1117')
plt.show()


### Chart 7 — Cumulative PnL Timeline (Coloured by Sentiment)

In [ ]:
daily = merged.sort_values('trade_date').groupby('trade_date').agg(
    cum_pnl   = ('Closed PnL', 'sum'),
    sentiment = ('classification', lambda x: x.mode()[0]),
).reset_index()
daily['cum_pnl']    = daily['cum_pnl'].cumsum() / 1e6
daily['trade_date'] = pd.to_datetime(daily['trade_date'])

fig, ax = plt.subplots(figsize=(14, 6), facecolor='#0D1117')
ax.set_facecolor('#161B22')

for i in range(len(daily) - 1):
    seg_color = SENTIMENT_COLORS.get(daily['sentiment'].iloc[i], '#95A5A6')
    ax.plot(daily['trade_date'].iloc[i:i+2],
            daily['cum_pnl'].iloc[i:i+2],
            color=seg_color, linewidth=1.8, solid_capstyle='round')

ax.fill_between(daily['trade_date'], daily['cum_pnl'], alpha=0.08, color='#58A6FF')
ax.axhline(0, color='#C9D1D9', linewidth=0.7, linestyle=':')

legend_patches = [mpatches.Patch(color=v, label=k) for k,v in SENTIMENT_COLORS.items()]
ax.legend(handles=legend_patches, fontsize=8.5, loc='upper left', framealpha=0.3)
ax.set_xlabel('Date', labelpad=8)
ax.set_ylabel('Cumulative PnL (Millions USD)', labelpad=8)
ax.set_title('Cumulative PnL Over Time — Coloured by Prevailing Market Sentiment', pad=10)
plt.xticks(rotation=20, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig('charts/chart7_cumulative_pnl_timeline.png', dpi=150, bbox_inches='tight', facecolor='#0D1117')
plt.show()


### Chart 8 — Cross-Margin vs Isolated-Margin: Win Rate & Avg PnL by Sentiment

In [ ]:
cross_eda    = merged[merged['Crossed']==True].groupby('classification', observed=True).agg(
    win_rate=('is_win','mean'), avg_pnl=('Closed PnL','mean')).reset_index()
isolated_eda = merged[merged['Crossed']==False].groupby('classification', observed=True).agg(
    win_rate=('is_win','mean'), avg_pnl=('Closed PnL','mean')).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(15, 6), facecolor='#0D1117')
fig.suptitle('Cross-Margin vs Isolated-Margin: Win Rate & Avg PnL by Sentiment',
             fontsize=14, color='#C9D1D9', y=1.01)

x, w = np.arange(len(SENTIMENT_ORDER)), 0.35
for ax, metric, ylabel, title in zip(
        axes,
        ['win_rate', 'avg_pnl'],
        ['Win Rate (%)', 'Avg Closed PnL (USD)'],
        ['Win Rate: Cross vs Isolated Margin', 'Avg PnL: Cross vs Isolated Margin']):
    ax.set_facecolor('#161B22')
    cv = [cross_eda.set_index('classification')[metric].get(c, 0) for c in SENTIMENT_ORDER]
    iv = [isolated_eda.set_index('classification')[metric].get(c, 0) for c in SENTIMENT_ORDER]
    if metric == 'win_rate':
        cv = [v*100 for v in cv]
        iv = [v*100 for v in iv]
    ax.bar(x - w/2, cv, w, label='Cross-margin',    color='#9B59B6', edgecolor='none', zorder=3)
    ax.bar(x + w/2, iv, w, label='Isolated-margin', color='#3498DB', edgecolor='none', zorder=3)
    ax.set_xticks(x)
    ax.set_xticklabels(SENTIMENT_ORDER, rotation=15, ha='right', fontsize=9)
    ax.set_ylabel(ylabel, labelpad=8)
    ax.set_title(title, pad=10)
    ax.legend(fontsize=9)
    ax.axhline(50 if metric=='win_rate' else 0,
               color='#F1C40F' if metric=='win_rate' else '#58A6FF',
               linewidth=1.2, linestyle='--', zorder=4)

plt.tight_layout()
plt.savefig('charts/chart8_cross_vs_isolated.png', dpi=150, bbox_inches='tight', facecolor='#0D1117')
plt.show()


## 6. Trader Archetype Deep-Dive

| Archetype | Behaviour | Profitability |
|-----------|-----------|---------------|
| **Mixed Sentiment** | Trades spread across all market states | Highest avg PnL (+$347K) — largest group, catches all opportunities |
| **Fear Opportunist** | Concentrates trades during low-sentiment windows | +$248K avg PnL — contrarian edge |
| **Consistent Winner** | Win rate >55%, well-diversified | +$226K — skill-based alpha |
| **FOMO Trader** | Heavy buyer during Extreme Greed | +$127K — benefits from momentum |
| **Greed Victim** | Avg sentiment >55 but nets losses | Negative — reactive, over-leveraged late entries |
| **Contrarian Buyer** | Buys aggressively during Extreme Fear | Rare archetype — high conviction, high risk |

### Key Findings
> **Fear Opportunists** outperform pure FOMO Traders by ~2x — selectively entering during market fear, when others capitulate, provides a structural edge.

> **Cross-margin traders** (using max available leverage) show a meaningfully different risk profile vs isolated-margin traders — see Chart 8 and the statistical test above.


## 7. Strategic Recommendation — Sentiment-Gated Trading Bot

Based on the data-driven insights above, a systematic trading algorithm on Hyperliquid should implement the following **Fear & Greed Index dynamic execution layer**:

---

### Rule Engine

```python
def sentiment_execution_filter(fg_score, fg_class, signal_side, base_position_size):
    """
    Dynamically adjusts position sizing and signal acceptance
    based on the current Fear & Greed Index reading.
    """
    multiplier = 1.0
    allow_trade = True

    if fg_class == 'Extreme Fear':
        # Low win rate (37%) — only accept high-conviction LONG signals
        # Block all SHORT entries (capitulation creates false sell signals)
        if signal_side == 'SELL':
            allow_trade = False
        else:
            multiplier = 0.6   # Reduce size — noisy environment

    elif fg_class == 'Fear':
        multiplier = 0.85      # Slight caution

    elif fg_class == 'Neutral':
        multiplier = 1.0       # Normal sizing

    elif fg_class == 'Greed':
        multiplier = 1.15      # Slightly scale up — positive momentum

    elif fg_class == 'Extreme Greed':
        # Highest avg PnL ($67.9) & best win rate (46.5%) in our data
        # Increase position size for BUY momentum trades
        if signal_side == 'BUY':
            multiplier = 1.40
        else:
            multiplier = 0.80  # SELL into extreme greed is risky

    final_size = base_position_size * multiplier
    return allow_trade, round(final_size, 2)
```

---

### Bot Architecture

```
[Daily Cron: Fetch FG Index]
         |
         v
[Store in Redis / DB as current_sentiment]
         |
         v
[Signal Generator (TA / ML Model)]
         |
         v
[sentiment_execution_filter(fg_score, fg_class, signal_side, base_size)]
         |
    allow_trade?
   /            \
YES              NO
 |               |
[Size position]  [Skip or queue for next session]
[Execute trade]
```

---

### Expected Impact
| Metric | Without Filter | With Filter (projected) |
|--------|----------------|-------------------------|
| Win Rate | ~41% (blended) | ~44-46% (+3–5pp) |
| Avg PnL/Trade | ~$47 | ~$55-65 |
| Max Drawdown | Uncontrolled | Reduced (blocks extreme-fear shorts) |
| Trade Volume | 100% | ~85% (filtered noise removed) |

> **Bottom line**: By skipping low-conviction Extreme Fear SELL signals and scaling up during confirmed Extreme Greed LONG opportunities, the bot converts market sentiment from a passive backdrop into an active risk management and position-sizing tool.


## 8. Export Summary Outputs

In [ ]:
eda.to_csv('eda_summary.csv', index=False)
trader.to_csv('trader_profiles.csv', index=False)
archetype_perf.to_csv('archetype_performance.csv', index=False)

print("Exported:")
print("  eda_summary.csv           — EDA metrics by sentiment")
print("  trader_profiles.csv       — Per-trader features, archetypes & cross-margin ratio")
print("  archetype_performance.csv — Archetype profitability + leverage usage table")
print()
print("Charts produced:")
for i, name in enumerate([
    'Win Rate & Avg PnL by Sentiment',
    'Trade Volume Distribution',
    'PnL Boxplot by Sentiment',
    'Directional Bias (BUY vs SELL)',
    'Archetype Profitability',
    'Sentiment vs PnL Scatter',
    'Cumulative PnL Timeline',
    'Cross vs Isolated Margin',
], 1):
    print(f"  Chart {i}: {name}")
